# CA4 Capstone Project #37: Hotel Revenue & Occupancy Analytics
**Course:** IBA – 5th Semester (Business Analytics)  
**Dataset:** Hotel Booking Demand Dataset ([Antonio, de Almeida, & Nunes, 2019, *Data in Brief*](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/) | [Kaggle Repository](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand))  
**Technology Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Streamlit

This notebook executes **Step 1 (Load & Understand)**, **Step 2 (Data Cleaning & Feature Engineering)**, **Step 3 (Exploratory Data Analysis)**, **Step 4 (Business Questions Analysis)**, and **Step 6–7 (Insights & Recommendations)**.

In [1]:
# STEP 1: LOAD AND UNDERSTAND THE DATASET
import zipfile, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists('hotel_bookings.csv'):
    df_raw = pd.read_csv('hotel_bookings.csv')
elif os.path.exists('hotel_bookings.csv.zip'):
    with zipfile.ZipFile('hotel_bookings.csv.zip', 'r') as z:
        with z.open('hotel_bookings.csv') as f:
            df_raw = pd.read_csv(f)

print(f'Raw Dataset Dimensions: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print('\nMissing Values Count:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print(f'\nExact Duplicate Rows: {df_raw.duplicated().sum():,}')
zero_guests = ((df_raw['adults'] + df_raw['children'].fillna(0) + df_raw['babies']) == 0).sum()
print(f'Zero-Guest Anomaly Rows (adults+children+babies == 0): {zero_guests:,}')

Raw Dataset Dimensions: 119,390 rows × 32 columns

Missing Values Count:
children         4
country        488
agent        16340
company     112593
dtype: int64

Exact Duplicate Rows: 31,994
Zero-Guest Anomaly Rows (adults+children+babies == 0): 180


In [2]:
# STEP 2: DATA CLEANING, PREPARATION & FEATURE ENGINEERING
df = df_raw.copy()

# 1. Impute missing values logically
df['children'] = df['children'].fillna(0).astype(int)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0).astype(int)
df['company'] = df['company'].fillna(0).astype(int)

# 2. Fix inconsistent category labels per data dictionary
df['meal'] = df['meal'].replace('Undefined', 'SC')
df['market_segment'] = df['market_segment'].replace('Undefined', 'Online TA')
df['distribution_channel'] = df['distribution_channel'].replace('Undefined', 'TA/TO')

# 3. Remove exact duplicates and invalid guest/ADR records
df_clean = df.drop_duplicates().copy()
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']
df_clean = df_clean[df_clean['total_guests'] > 0].copy()
df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] <= 1000)].copy()

# 4. Create Calculated Business Features
df_clean['total_stay_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['booking_revenue'] = df_clean['adr'] * df_clean['total_stay_nights']
df_clean['realized_revenue'] = np.where(df_clean['is_canceled'] == 0, df_clean['booking_revenue'], 0.0)
df_clean['lost_revenue'] = np.where(df_clean['is_canceled'] == 1, df_clean['booking_revenue'], 0.0)
df_clean['is_room_upgraded'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)
df_clean['lead_time_bucket'] = pd.cut(
    df_clean['lead_time'],
    bins=[-1, 7, 30, 90, 180, 800],
    labels=['0–7d (Last-Min)', '8–30d (Short)', '31–90d (Medium)', '91–180d (Long)', '180+d (Very Long)']
)

print('Cleaned Deduplicated Dataset Shape:', df_clean.shape)
print(f"Overall Unique Cancellation Rate: {df_clean['is_canceled'].mean()*100:.2f}%")
print(f"Gross Room Revenue Pipeline: €{df_clean['booking_revenue'].sum():,.2f}")
print(f"Realized Room Revenue (Check-Out): €{df_clean['realized_revenue'].sum():,.2f}")
print(f"Lost Room Revenue (Canceled): €{df_clean['lost_revenue'].sum():,.2f}")

Cleaned Deduplicated Dataset Shape: (87228, 40)
Overall Unique Cancellation Rate: 27.52%
Gross Room Revenue Pipeline: €34,446,254.93
Realized Room Revenue (Check-Out): €22,967,611.95
Lost Room Revenue (Canceled): €11,478,642.98


In [3]:
# STEP 3 & 4: EXPLORATORY DATA ANALYSIS & BUSINESS QUESTIONS
prop_summary = df_clean.groupby('hotel').agg(
    bookings=('is_canceled', 'count'),
    cancel_rate=('is_canceled', lambda x: round(x.mean()*100, 2)),
    avg_adr=('adr', lambda x: round(x.mean(), 2)),
    avg_stay=('total_stay_nights', lambda x: round(x.mean(), 2)),
    realized_rev=('realized_revenue', lambda x: round(x.sum(), 2)),
    lost_rev=('lost_revenue', lambda x: round(x.sum(), 2))
).reset_index()
print('--- Property Performance Summary ---')
print(prop_summary.to_string(index=False))

seg_summary = df_clean.groupby('market_segment').agg(
    bookings=('is_canceled', 'count'),
    cancel_rate=('is_canceled', lambda x: round(x.mean()*100, 2)),
    avg_adr=('adr', lambda x: round(x.mean(), 2)),
    realized_rev=('realized_revenue', lambda x: round(x.sum(), 2)),
    lost_rev=('lost_revenue', lambda x: round(x.sum(), 2))
).sort_values('bookings', ascending=False).head(5).reset_index()
print('\n--- Market Segment Performance Summary ---')
print(seg_summary.to_string(index=False))

--- Property Performance Summary ---
        hotel  bookings  cancel_rate  avg_adr  avg_stay  realized_rev    lost_rev
   City Hotel     53273        30.10   111.17      3.14   12150577.73  6609055.63
 Resort Hotel     33955        23.48    99.06      4.39   10817034.22  4869587.35

--- Market Segment Performance Summary ---
 market_segment  bookings  cancel_rate  avg_adr  realized_rev    lost_rev
      Online TA     51555        35.39   118.29   12966424.61  9292949.94
  Offline TA/TO     13854        14.84    81.57    4595456.13   815608.82
         Direct     11780        14.75   116.78    3925328.52   929714.23
         Groups      4921        27.07    75.18     922900.97   315619.75
      Corporate      4200        12.12    68.34     483728.83   110415.25


In [4]:
# LEAD TIME & SPECIAL REQUESTS RISK ANALYSIS
lead_summary = df_clean.groupby('lead_time_bucket', observed=False).agg(
    bookings=('is_canceled', 'count'),
    cancel_rate=('is_canceled', lambda x: round(x.mean()*100, 2)),
    avg_adr=('adr', lambda x: round(x.mean(), 2)),
    realized_rev=('realized_revenue', lambda x: round(x.sum(), 2)),
    lost_rev=('lost_revenue', lambda x: round(x.sum(), 2))
).reset_index()
print('--- Lead Time Window Analysis ---')
print(lead_summary.to_string(index=False))

--- Lead Time Window Analysis ---
   lead_time_bucket  bookings  cancel_rate  avg_adr  realized_rev    lost_rev
   0–7d (Last-Min)     18203         8.42    91.68    3147980.29   385212.81
     8–30d (Short)     16326        25.39   110.70    3995279.51  1767350.55
   31–90d (Medium)     22719        32.04   111.11    6263696.67  3305095.17
    91–180d (Long)     18224        35.01   114.17    5960993.98  3462951.05
 180+d (Very Long)     11756        39.74   102.51    3599661.50  2558033.40
